##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

###Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max, count, avg, when
from pyspark.sql import Window
import pyspark.sql.functions as F

### Products Table Data Manipulation and Cleaning

In [0]:
df_products_bronze = spark.table("olist_ecommerce_project.bronze.brz_products")

# Basic profiling
print("Total rows:", df_products_bronze.count())
print("Distinct product_id:", df_products_bronze.select("product_id").distinct().count())

# Null check
df_products_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_products_bronze.columns
]).show()

Let's confirm our assumption that the 610 nulls are all the same rows and the 2 dimension nulls are different rows

In [0]:
# Confirm 610 nulls are the same rows
print("Rows where ALL text metrics are null together:")
df_products_bronze.filter(
    col("product_category_name").isNull() &
    col("product_name_length").isNull() &
    col("product_description_length").isNull()
).count()



In [0]:
# Confirm the 2 dimension nulls
print("\nRows where dimensions are null:")
df_products_bronze.filter(
    col("product_weight_g").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).show(truncate=False)

In [0]:
# Check if the 610 null category rows have dimension values
df_products_bronze.filter(
    col("product_category_name").isNull()
).select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm"
).show(10, truncate=False)

Fixing the null values

In [0]:

# Step 1: Fill null category with 'unknown' instead of dropping
df_products_clean = df_products_bronze.withColumn(
    "product_category_name",
    when(col("product_category_name").isNull(), "unknown")
    .otherwise(col("product_category_name"))
)

# Verify no more null categories
print("Null categories remaining:", df_products_clean.filter(col("product_category_name").isNull()).count())
print("Total rows:", df_products_clean.count())

# Check unknown category count
print("Unknown category rows:", df_products_clean.filter(col("product_category_name") == "unknown").count())

Join the Category translation as we have a table where product have a translation in English

In [0]:
# Load category translation table
df_category = spark.table("olist_ecommerce_project.bronze.brz_category_name")

# Join to add English category names
df_products_silver = df_products_clean.join(
    df_category.select("product_category_name", "product_category_name_english"),
    on="product_category_name",
    how="left"
)

# Sanity check
print("Rows after join:", df_products_silver.count())

# Check if 'unknown' category got a translation (it won't, should be null)
df_products_silver.filter(
    col("product_category_name") == "unknown"
).select(
    "product_category_name",
    "product_category_name_english"
).show(3, truncate=False)

# Check a few normal rows to confirm translation worked
df_products_silver.filter(
    col("product_category_name") != "unknown"
).select(
    "product_category_name",
    "product_category_name_english"
).show(10, truncate=False)

In [0]:
# Step 3: Fill null English category for 'unknown' products
df_products_silver = df_products_silver.withColumn(
    "product_category_name_english",
    when(col("product_category_name_english").isNull(), "unknown")
    .otherwise(col("product_category_name_english"))
)

# Step 4: Drop source_file audit column
df_products_silver = df_products_silver.drop("_source_file")

# Final sanity check before writing
print("Final rows:", df_products_silver.count())
df_products_silver.select(
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_weight_g"
).show(10, truncate=False)

In [0]:
display(df_products_silver.limit(20))

##### Creating the Products Silver Table

In [0]:
(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_products")
)

print("slv_products written successfully")